In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate
import copy
from numpy import unique
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Embedding
from keras.layers import concatenate
from keras.utils import plot_model
import math
from tensorflow.keras.layers import Reshape

In [11]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
#df = pd.read_pickle("Data/df_preprocessed_train.pickle")
df_train_original = pd.read_pickle("Data/df_preprocessed_no_encoding_train.pickle")
df_test_original = pd.read_pickle("Data/df_preprocessed_no_encoding_test.pickle")

In [9]:
df_train = copy.deepcopy(df_train_original[:10000])
df_test = copy.deepcopy(df_test_original[:2000])

In [5]:
#let's one-hot-encode for procedure, type contract, central purchasing, eu_fund, framework or dynamic purchasing, 
base_n_encoder_cols = ["cpv_code", "country", "language"]
one_hot_columns = ["type_contract", "procedure", "joint_procurement_involved", "central_purchasing", "eu_fund", "framework or dynamic purchasing", "contracting authority type", "contracting authority activity"]
numerical_columns = ["duration", "nb_tenders_received", "nb_tenders_received_sme"]
text_embedding_columns = ["short_descr", "ac criteria", "object_contract/title", "object descr titles", "ac cost criteria"]
categorical_columns_original = base_n_encoder_cols + one_hot_columns

In [6]:
column_order = categorical_columns_original + numerical_columns + ["award_contract/val_total: 0"]

In [10]:
#initialize the data
drop_columns = ["original index", "date_conclusion_contract"] + text_embedding_columns
df_train = df_train.drop(columns = drop_columns)
df_test = df_test.drop(columns = drop_columns)

df_train = df_train[column_order]
df_test = df_test[column_order]

X_train = df_train[[col for col in df_train.columns if col != "award_contract/val_total: 0"]].to_numpy()
y_train = df_train["award_contract/val_total: 0"].to_numpy()

X_test = df_test[[col for col in df_train.columns if col != "award_contract/val_total: 0"]].to_numpy()
y_test = df_test["award_contract/val_total: 0"].to_numpy()

x_total = np.append(X_train, X_test)

In [68]:
#prepare datainput of categorical columns before embedding
def encoder(df, list_of_columns):
    unique_values_per_col = {}
    label_encoder = LabelEncoder()

    for col in list_of_columns:
        df[col] = label_encoder.fit_transform(df[col])
        unique_values_per_col[col] = list(label_encoder.classes_)

    return unique_values_per_col

In [69]:
unique_values_per_col_train = encoder(df_train, categorical_columns_original)
unique_values_per_col_test = encoder(df_test, categorical_columns_original)
unique_values_per_col = {"train": unique_values_per_col_train, "test": unique_values_per_col_test}

In [70]:
input_cat = []
emb_cat = []
embedding_dim_text = 384

#create embeddings for the categorical columns
for col in unique_values_per_col["train"].keys():
    #get the number of labels per col
    n_labels = len(np.unique(unique_values_per_col["train"][col]))
    #define the input layer for the i-th categorical feature
    input_layer = Input(shape=(1,))
    #define the embedding layer
    embedding_layer = Embedding(input_dim = n_labels, output_dim = math.ceil(math.sqrt(n_labels)))(input_layer)
    input_cat.append(input_layer)
    emb_cat.append(embedding_layer)